# SimpleTokenizerV3

- This is an optional tokenizer analogous to `SimpleTokenizerV2` in the main chapter
- This `SimpleTokenizerV3` one keeps delimiters and whitespace as tokens, which allows a more natural round trip to preserve the original text

&nbsp;
## SimpleTokenizerV2 behavior

- Note that `SimpleTokenizerV2` removes whitespace tokens, so it can't recover repeated spaces or line breaks during decoding

In [1]:
import re


class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {index: token for token, index in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int
            else '<|unk|>' for item in preprocessed
        ]
        return [self.str_to_int[item] for item in preprocessed]

    def decode(self, ids):
        text = ' '.join(self.int_to_str[index] for index in ids)
        return re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)

In [2]:
training_text = 'I, HAD tea.\nI-- HAD tea.'
v2_tokens = re.split(r'([,.:;?_!"()\']|--|\s)', training_text)
v2_tokens = [item.strip() for item in v2_tokens if item.strip()]
v2_vocab = {token: index for index, token in enumerate(sorted(set(v2_tokens)) + ['<|unk|>'])}
tokenizer_v2 = SimpleTokenizerV2(v2_vocab)

sample_text = 'I,  HAD tea.\nI-- HAD tea.'
decoded_text = tokenizer_v2.decode(tokenizer_v2.encode(sample_text))
assert decoded_text != sample_text

print('Original:', repr(sample_text))
print('Decoded: ', repr(decoded_text))

Original: 'I,  HAD tea.\nI-- HAD tea.'
Decoded:  'I, HAD tea. I -- HAD tea.'


&nbsp;
## SimpleTokenizerV3 behavior

- Now, the `SimpleTokenizerV3` keeps the whitespace tokens so that decoding can restore the original text

In [3]:
class SimpleTokenizerV3:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {index: token for token, index in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item for item in preprocessed if item != '']
        return [self.str_to_int.get(item, self.str_to_int['<|unk|>']) for item in preprocessed]

    def decode(self, ids):
        return ''.join(self.int_to_str[index] for index in ids)

In [4]:
v3_tokens = re.split(r'([,.:;?_!"()\']|--|\s)', training_text)
v3_vocab = {token: index for index, token in enumerate(sorted(set(token for token in v3_tokens if token)) + ['<|unk|>'])}
tokenizer_v3 = SimpleTokenizerV3(v3_vocab)
decoded_text = tokenizer_v3.decode(tokenizer_v3.encode(sample_text))
assert decoded_text == sample_text

print('Original:', repr(sample_text))
print('Decoded: ', repr(decoded_text))

for sample in ('I-- HAD', 'I, HAD', 'I  HAD', 'I\nHAD'):
    assert tokenizer_v3.decode(tokenizer_v3.encode(sample)) == sample

assert tokenizer_v3.str_to_int['<|unk|>'] in tokenizer_v3.encode('unknownword')

Original: 'I,  HAD tea.\nI-- HAD tea.'
Decoded:  'I,  HAD tea.\nI-- HAD tea.'
